In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.." 
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
RUN = f"{REPO}/results/20260816_230437_38678/csv"
canon = pd.read_csv(f"{RUN}/canonical_traces.csv")
quant_induced = pd.read_csv(f"{RUN}/quant_induced_summary.csv")
quant_induced

,model,dataset,spearman_fp32_ptq,spearman_fp32_qat,conv1_fp32,conv1_ptq,conv1_ptq_amplification,conv1_amp_rank,top_amplified_layer,banked_fp32_matches,note
0,resnet50_no_weights,CIFAR10,0.904784,0.891519,14.904178,97.761784,0.560263,51,layer4.2.conv3,not_provided,NaN
1,resnet18_no_weights,CIFAR10,0.658442,0.671429,17.039128,173.218405,0.951243,19,fc,not_provided,NaN
2,cnn,CIFAR10,NaN,NaN,63.149640,530.117086,1.151272,6,conv4,not_provided,NaN


In [ ]:
agg = canon[(canon["dataset"] == "CIFAR10") & (canon["seed"] == "aggregate") & (canon["canonical_layer"] == "conv1")]
fused = agg[agg["variant"] == "fp32_fused"].set_index("model")["trace_raw"]
unfused = agg[agg["variant"] == "fp32_unfused"].set_index("model")["trace_raw"]
fusion_ratio = (fused / unfused).rename("fusion_ratio")
fusion_ratio

model
resnet50_no_weights    11.707636
resnet18_no_weights    10.686986
cnn                     7.291601
Name: fusion_ratio, dtype: float64

In [ ]:
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}

qi = quant_induced.set_index("model").copy()
qi["fusion_ratio"] = fusion_ratio
qi["Modell"] = [MODEL_LABEL[m] for m in qi.index]

qi["Verdikt"] = np.where(qi["conv1_ptq_amplification"] > 1.0, "quant_induced", "fp32_intrinsic")

table5 = qi[["Modell", "fusion_ratio", "conv1_ptq_amplification", "conv1_amp_rank", "Verdikt"]].copy()
table5.columns = ["Modell", "Fusion", "PTQ-Amp.", "Rang", "Verdikt"]
table5["Fusion"] = table5["Fusion"].round(2)
table5["PTQ-Amp."] = table5["PTQ-Amp."].round(2)
n_layers = {"cnn": 6, "resnet18_no_weights": 21, "resnet50_no_weights": 54}
table5["Rang"] = [f"{r}/{n_layers[m]}" for m, r in zip(qi.index, table5["Rang"])]
table5.to_csv(f"{FIG_DIR}/tab_05_fusion_vs_quantization.csv", index=False)
table5

,Modell,Fusion,PTQ-Amp.,Rang,Verdikt
model,,,,,
resnet50_no_weights,ResNet-50,11.71,0.56,51/54,fp32_intrinsic
resnet18_no_weights,ResNet-18,10.69,0.95,19/21,fp32_intrinsic
cnn,CNN,7.29,1.15,6/6,quant_induced
